# Anomaly Detection

Identifying rare, unusual observations that don't conform to expected patterns.

1. **Isolation Forest** - Tree-based anomaly detection
2. **Local Outlier Factor (LOF)** - Density-based
3. **One-Class SVM** - Boundary-based
4. **Comparison** on synthetic and real data

**Dataset**: Synthetic + Credit Card Fraud (simulated)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

In [ ]:
# Generate normal data + outliers
np.random.seed(42)
X_normal, _ = make_blobs(n_samples=300, centers=1, cluster_std=1.0, random_state=42)
X_outliers = np.random.uniform(low=-6, high=6, size=(30, 2))
X = np.vstack([X_normal, X_outliers])
y_true = np.array([1] * 300 + [-1] * 30)  # 1=inlier, -1=outlier

print(f"Total samples: {len(X)}, Outliers: {(y_true == -1).sum()} ({(y_true == -1).mean()*100:.0f}%)")

In [ ]:
# Compare methods
detectors = {
    "Isolation Forest": IsolationForest(contamination=0.1, random_state=42),
    "Local Outlier Factor": LocalOutlierFactor(contamination=0.1),
    "One-Class SVM": OneClassSVM(nu=0.1, gamma="scale"),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, detector) in zip(axes, detectors.items()):
    if name == "Local Outlier Factor":
        y_pred = detector.fit_predict(X)
    else:
        y_pred = detector.fit_predict(X)
    
    inliers = y_pred == 1
    outliers = y_pred == -1
    
    ax.scatter(X[inliers, 0], X[inliers, 1], c="steelblue", s=15, label="Inlier")
    ax.scatter(X[outliers, 0], X[outliers, 1], c="red", s=25, marker="x", label="Outlier")
    
    # Accuracy
    correct = (y_pred == y_true).sum()
    ax.set_title(f"{name} ({correct}/{len(y_true)} correct)")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Isolation Forest: anomaly scores
iso = IsolationForest(contamination=0.1, random_state=42)
iso.fit(X)
scores = iso.decision_function(X)  # lower = more anomalous

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

scatter = axes[0].scatter(X[:, 0], X[:, 1], c=scores, cmap="RdYlBu", s=15)
plt.colorbar(scatter, ax=axes[0], label="Anomaly Score")
axes[0].set_title("Isolation Forest: Anomaly Scores")

axes[1].hist(scores[:300], bins=30, alpha=0.7, label="Inliers", color="steelblue")
axes[1].hist(scores[300:], bins=15, alpha=0.7, label="Outliers", color="red")
axes[1].set_xlabel("Anomaly Score")
axes[1].set_ylabel("Count")
axes[1].set_title("Score Distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

## Key Takeaways

1. **Isolation Forest** is the most versatile - works well in high dimensions, fast
2. **LOF** excels at detecting local anomalies in varying-density data
3. **One-Class SVM** learns a boundary around normal data - good with clear separation
4. **Contamination parameter** is critical - set it based on domain knowledge of expected anomaly rate
5. **Anomaly scores > binary labels** - use thresholds based on business requirements